# Phase 4: Wave Classifier — ResNet18 with SWA + CutMix

This notebook trains a ResNet18 transfer-learning model to classify wave drawings as **Healthy** or **Parkinson's**.

**Dataset:**
- `data/wave/training/` → 36 healthy + 36 parkinson images (72 total)
- `data/wave/testing/`  → 15 healthy + 15 parkinson images (30 total)

---

## v2 Strategy — Different from Spiral

Wave uses a more aggressive training strategy than Spiral. Key differences:

| | Spiral (Phase 3) | Wave (Phase 4) |
|---|---|---|
| Unfrozen layers | layer4 only | **layer3 + layer4** |
| Head | 512→128→1 | **512→256→64→1 (GELU)** |
| Augmentation | MixUp only | **MixUp + CutMix** |
| Training epochs | 60 | **80** |
| Special technique | Multi-seed ensemble | **Stochastic Weight Averaging (SWA)** |

### Why CutMix works for Wave but not Spiral:
Wave drawings are **periodic** — the tremor irregularity repeats across the entire image.
Any patch cut from a wave image still contains diagnostic information.
For spiral, the signal is the **continuous radial structure** — cutting destroys it.

### Stochastic Weight Averaging (SWA):
From epoch 50 onwards, SWA averages the model weights every epoch.
This acts like a free ensemble — the averaged weights sit in a flatter loss region
that generalizes better than any single checkpoint.

**Final Result: 93.33% standard accuracy (28/30 test images correct)**

**Saved to:** `backend/models/wave_resnet18.pth`

In [ ]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# ==========================================
# CONFIG
# ==========================================
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS    = 80
WARMUP_EP = 5     # linear LR warmup epochs
SWA_START = 50    # epoch at which SWA begins averaging
BATCH     = 8
BASE_LR   = 1e-4
CUTMIX_PROB = 0.5  # probability of using CutMix vs MixUp each batch
TRAIN_DIR = '../data/wave/training'
TEST_DIR  = '../data/wave/testing'
SAVE_PATH = '../backend/models/wave_resnet18.pth'

print(f'Device: {device}')
print(f'Training for {EPOCHS} epochs | SWA starts at epoch {SWA_START}')

In [ ]:
# ==========================================
# STEP 2: DATA AUGMENTATION
# ==========================================
# Full aggressive augmentation stack for wave:
#   - RandomErasing: masks patches to prevent texture bias
#   - MixUp + CutMix applied during training loop (not here)

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load datasets
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
test_ds  = datasets.ImageFolder(TEST_DIR,  transform=test_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

print(f'Train: {len(train_ds)} images | Test: {len(test_ds)} images')
print(f'Classes: {train_ds.class_to_idx}')

In [ ]:
# ==========================================
# STEP 3: HELPER FUNCTIONS
# ==========================================

def mixup_data(x, y, alpha=0.3):
    """MixUp: linearly blend two images and their labels."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def cutmix_data(x, y, alpha=1.0):
    """CutMix: cut a rectangle from one image and paste into another.
    lambda = fraction of ORIGINAL image that remains uncut.
    Works well for wave (periodic signal) — bad for spiral (continuous radial).
    """
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    _, _, H, W = x.shape
    cut_ratio = np.sqrt(1.0 - lam)
    cut_h, cut_w = int(H * cut_ratio), int(W * cut_ratio)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, y1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    x2, y2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (H * W)  # recalculate from actual box
    return mixed_x, y, y[idx], lam


def label_smooth_bce(pred, target, smooth=0.1):
    target_s = target * (1 - smooth) + 0.5 * smooth
    return nn.BCEWithLogitsLoss()(pred, target_s)


def mixed_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def freeze_bn(model):
    """Freeze all BatchNorm layers.
    ImageNet BN statistics are better than recalculated stats from 72 images.
    Must be called after model.train() since train() re-enables BN.
    """
    for module in model.modules():
        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d)):
            module.eval()
            module.weight.requires_grad = False
            module.bias.requires_grad   = False

print('Helper functions defined.')

In [ ]:
# ==========================================
# STEP 4: BUILD MODEL
# ==========================================
# Wave model: unfreeze layer3 + layer4 (more aggressive than spiral).
# Wider head: 512 -> 256 -> 64 -> 1 with GELU activation.
# GELU is smoother than ReLU and often performs better on pre-trained networks.
#
# Raw logits output — Sigmoid added at inference in backend/main.py.
# Training uses BCEWithLogitsLoss (numerically more stable than BCE + Sigmoid).

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze all layers first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer3 AND layer4
for name, child in model.named_children():
    if name in ('layer3', 'layer4'):
        for param in child.parameters():
            param.requires_grad = True

# Wider head with GELU
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 256),
    nn.GELU(),
    nn.Dropout(0.4),
    nn.Linear(256, 64),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(64, 1)   # raw logit
)

model = model.to(device)
freeze_bn(model)  # Freeze BN after moving to device

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,}')

In [ ]:
# ==========================================
# STEP 5: TRAINING WITH SWA
# ==========================================

# SWA wraps the base model and maintains a running average of its weights
swa_model = AveragedModel(model)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=BASE_LR, weight_decay=1e-4
)

# Phase 1: linear warmup (epochs 1-5)
warmup_sched = optim.lr_scheduler.LambdaLR(
    optimizer, lambda ep: float(ep + 1) / WARMUP_EP if ep < WARMUP_EP else 1.0
)
# Phase 2: cosine annealing after warmup
cosine_sched = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=(EPOCHS - WARMUP_EP), eta_min=1e-7
)
# Phase 3: SWA uses a constant low LR during averaging
swa_scheduler = SWALR(optimizer, swa_lr=5e-6)

best_val_loss = float('inf')
best_weights  = None
swa_active    = False

for epoch in range(1, EPOCHS + 1):
    # Keep BN frozen (model.train() re-enables it)
    model.train()
    freeze_bn(model)
    running_loss = 0.0

    for imgs, labels in train_loader:
        imgs   = imgs.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        # Randomly choose CutMix or MixUp each batch
        if np.random.rand() < CUTMIX_PROB:
            mixed, y_a, y_b, lam = cutmix_data(imgs, labels, alpha=1.0)
        else:
            mixed, y_a, y_b, lam = mixup_data(imgs, labels, alpha=0.3)

        optimizer.zero_grad()
        logits = model(mixed)
        loss = mixed_loss(
            lambda p, t: label_smooth_bce(p, t, smooth=0.1),
            logits, y_a, y_b, lam
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()

    # Scheduler step
    if epoch <= WARMUP_EP:
        warmup_sched.step()
    elif epoch < SWA_START:
        cosine_sched.step()
    else:
        if not swa_active:
            swa_active = True
            print(f'  [SWA] Weight averaging started at epoch {epoch}')
        swa_model.update_parameters(model)
        swa_scheduler.step()

    # Validation (on base model)
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs   = imgs.to(device)
            labels = labels.to(device).float().unsqueeze(1)
            logits = model(imgs)
            val_loss += label_smooth_bce(logits, labels).item()
            preds    = (torch.sigmoid(logits) > 0.5).float()
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)

    val_acc   = correct / total
    avg_tloss = running_loss / len(train_loader)
    avg_vloss = val_loss     / len(test_loader)

    if avg_vloss < best_val_loss:
        best_val_loss = avg_vloss
        best_weights  = copy.deepcopy(model.state_dict())
        marker = ' << best'
    else:
        marker = ''

    if epoch % 5 == 0 or epoch == 1:
        swa_tag = ' [SWA]' if swa_active else ''
        print(f'  Epoch {epoch:3d}/{EPOCHS} | '
              f'train={avg_tloss:.4f} | val={avg_vloss:.4f} | '
              f'acc={val_acc*100:.1f}%{marker}{swa_tag}')

print('\nTraining complete!')

In [ ]:
# ==========================================
# STEP 6: SWA BN UPDATE + FINAL EVALUATION
# ==========================================
# SWA model has averaged weights but outdated BN statistics.
# update_bn() re-computes BN running stats from the training data.

print('Updating SWA BatchNorm statistics...')
update_bn(train_loader, swa_model, device=device)

# Evaluate SWA model
swa_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs   = imgs.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        preds  = (torch.sigmoid(swa_model(imgs)) > 0.5).float()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
swa_acc = correct / total
print(f'SWA Model Accuracy: {swa_acc*100:.2f}%')

# Evaluate best-checkpoint model
model.load_state_dict(best_weights)
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs   = imgs.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        preds  = (torch.sigmoid(model(imgs)) > 0.5).float()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
ckpt_acc = correct / total
print(f'Checkpoint Model Accuracy: {ckpt_acc*100:.2f}%')

# Save whichever is better
if swa_acc >= ckpt_acc:
    print(f'\n[save] Using SWA model ({swa_acc*100:.1f}%)')
    # Extract SWA state dict (strip 'module.' prefix from AveragedModel)
    save_state = {k.replace('module.', ''): v
                  for k, v in swa_model.state_dict().items()
                  if 'n_averaged' not in k}
    torch.save(save_state, SAVE_PATH)
else:
    print(f'\n[save] Using Checkpoint model ({ckpt_acc*100:.1f}%)')
    torch.save(model.state_dict(), SAVE_PATH)

print(f'[saved] {SAVE_PATH}')
print(f'\nFinal Wave Accuracy: {max(swa_acc, ckpt_acc)*100:.2f}%')